In [ ]:
import numpy as np
import pandas as pd

### Loading Data

In [ ]:
train = pd.read_csv('../Data/train.csv')
test = pd.read_csv('../Data/test.csv')

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.info()

In [ ]:
test.info()

### Data preprocessing

In [ ]:
train.isna().sum()

In [ ]:
test.isna().sum()

In [ ]:
train.duplicated().sum()

In [ ]:
train.select_dtypes(include = 'number').columns

In [ ]:
train.select_dtypes(include = 'object').columns

In [ ]:
test.select_dtypes(include = 'number').columns

In [ ]:
test.select_dtypes(include = 'object').columns

### Dropping unnecessary columns

In [ ]:
train.drop(columns = ['id'], inplace = True)

In [ ]:
test_ids = test['id']
test.drop(columns = ['id'], inplace = True)

### Splitting data

In [ ]:
X = train.drop(columns = ['class'])
y = train['class']

In [ ]:
num_cols = X.select_dtypes(include = 'number').columns.tolist()
cat_cols = X.select_dtypes(include = 'object').columns.tolist()

In [ ]:
cat_cols

In [ ]:
for i in cat_cols:
    print(f'{i} Unique Values: {train[i].unique()}')

In [ ]:
# we need to split the cat_cols further more 
ohe_cols = [cat_cols[0]]
ord_cols = [cat_cols[1]]

### Creating Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder

In [ ]:
num_pipe = Pipeline(steps = [
    ('Scaling', StandardScaler())
])

ohe_pipe = Pipeline(steps = [
    ('OneHotEncoder', OneHotEncoder())
])

ord_pipe = Pipeline(steps = [
    ('OrdinalEncoder', OrdinalEncoder())
])

In [ ]:
preprocessor = ColumnTransformer(transformers = [
    ('numerical transformation', num_pipe, num_cols),
    ('one hot transformation', ohe_pipe, ohe_cols),
    ('ordinal transformation', ord_pipe, ord_cols)
])

In [ ]:
X_transformed = preprocessor.fit_transform(X)
test_transformed = preprocessor.transform(test)

### Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, recall_score, balanced_accuracy_score

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_transformed, y, test_size = 0.2, random_state = 42)

In [ ]:
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(),
    'RandomForestClassifier': RandomForestClassifier(),
    'XGBClassifier': XGBClassifier(),
    'CatboostClassifier': CatBoostClassifier(verbose = False)
}

In [ ]:
best_model = None
best_score = 0
model_name = ''

for name, model in models.items():
    print('Model name: ', name)
    model.fit(X_train, y_train_enc)
    y_pred = model.predict(X_test)
    print('Metrics for', name)
    print('F1 score: ', f1_score(y_test_enc, y_pred, average = 'macro'))
    print('Recall score: ', recall_score(y_test_enc, y_pred, average = 'macro'))
    print('Balanced accuracy score: ', balanced_accuracy_score(y_test_enc, y_pred))
    print('*' * 100)

    score = balanced_accuracy_score(y_test_enc, y_pred)
    if score > best_score:
        best_score = score
        best_name = name
        best_model = model

    print(f'Best model: {name}, with balanced accuracy score: {best_score}')

In [ ]:
results = model.predict(test_transformed)

In [ ]:
results = le.inverse_transform(results)

In [ ]:
results

### Saving Result

In [ ]:
subbmission = pd.DataFrame({
    'id': test_ids,
    'class': results
    })

subbmission.to_csv('../Data/submission.csv', index = False)